# Titanic Survival Prediction

This notebook builds a complete passenger survival classifier from the local `Titanic.csv` dataset. It explores the data, prepares features without leaking information from the validation set, trains and evaluates a scikit-learn pipeline, then saves a final model for the project website.

**Run from the project folder** (`Titanic Survival Prediction`) with the project `.venv` kernel selected. Run cells from top to bottom. The reported metrics come from a held-out test split; the saved demo model is refit on all available rows.


## 1. Import libraries

Pandas handles tabular data, Matplotlib makes the exploratory charts, and scikit-learn provides preprocessing, model training, and evaluation. Joblib writes the trained pipeline to disk.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("Titanic.csv")
MODEL_PATH = Path("titanic_survival_pipeline.joblib")


## 2. Load and check the data

The CSV is expected beside this notebook. We check the columns required by the model before continuing. The unnamed index and `alive` label are excluded from model inputs; `alive` duplicates the target and would leak the answer.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Run this notebook from the project folder."
    )

df = pd.read_csv(DATA_PATH)
required_columns = {"survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head()

## 3. Understand the columns

Review data types, ranges, and summary statistics. The target is `survived` (`1` means survived, `0` means did not survive).

In [ ]:
df.info()
df.describe(include="all").T

## 4. Check data quality

Missing values are handled inside the model pipeline: numeric values use the training median, while categories use the most frequent training value. This ensures imputation is learned from training data only.

In [ ]:
missing = (
    df.isna().sum()
      .rename("missing_count")
      .to_frame()
      .assign(missing_percent=lambda table: (table["missing_count"] / len(df) * 100).round(1))
      .sort_values("missing_count", ascending=False)
)
missing

## 5. Explore the outcome

First inspect the class balance. Accuracy alone can be misleading when one target class is much more common than another.

In [ ]:
survival_counts = df["survived"].value_counts().reindex([0, 1], fill_value=0)
ax = survival_counts.plot(kind="bar", color=["#455a64", "#c9a96e"], figsize=(6, 4))
ax.set_xticklabels(["Did not survive", "Survived"], rotation=0)
ax.set_ylabel("Passengers")
ax.set_title("Passenger survival counts")
plt.tight_layout()
plt.show()
print("Overall survival rate:", f"{df['survived'].mean():.1%}")

## 6. Explore survival by sex and passenger class

These group averages are descriptive patterns in this dataset. They do not establish cause and should not be interpreted as guarantees for an individual passenger.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
(df.groupby("sex")["survived"].mean().sort_values(ascending=False)
   .plot(kind="bar", ax=axes[0], color="#c9a96e"))
axes[0].set_title("Survival rate by sex")
axes[0].set_ylabel("Survival rate")
axes[0].tick_params(axis="x", rotation=0)
(df.groupby("pclass")["survived"].mean().sort_index()
   .plot(kind="bar", ax=axes[1], color="#607d8b"))
axes[1].set_title("Survival rate by ticket class")
axes[1].set_ylabel("Survival rate")
axes[1].set_xlabel("Passenger class")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 7. Explore age and fare

The distributions help reveal skew and missingness, which informs the choice of median imputation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["age"].dropna().plot(kind="hist", bins=25, ax=axes[0], color="#c9a96e", edgecolor="white")
axes[0].set(title="Passenger age distribution", xlabel="Age")
df["fare"].dropna().plot(kind="hist", bins=30, ax=axes[1], color="#607d8b", edgecolor="white")
axes[1].set(title="Fare distribution", xlabel="Fare")
plt.tight_layout()
plt.show()

## 8. Create a family-size feature

`family_size` combines siblings/spouses and parents/children aboard, plus the passenger. This feature matches the input schema expected by the project's web app.

In [ ]:
df = df.copy()
df["family_size"] = df["sibsp"] + df["parch"] + 1
df[["sibsp", "parch", "family_size", "survived"]].head()

## 9. Select model inputs and target

We use class, sex, age, family size, fare, and port of embarkation. The identifiers, names, ticket text, cabin, and `alive` are not used. The pipeline handles missing age/embarkation values.

In [ ]:
FEATURES = ["pclass", "sex", "age", "family_size", "fare", "embarked"]
TARGET = "survived"
X = df[FEATURES].copy()
y = df[TARGET].astype(int)

print("Model features:", FEATURES)
print("Target values:", sorted(y.unique()))

## 10. Split into training and test sets

The test set is held aside for evaluation. Stratification keeps the survival/non-survival proportions similar in both sets. A fixed random seed makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

## 11. Build preprocessing steps

Numeric features receive median imputation and scaling. Categorical features receive most-frequent imputation and one-hot encoding. `ColumnTransformer` keeps these steps tied to the correct columns, and the full pipeline ensures the same transformations are applied at prediction time.

In [ ]:
NUMERIC_FEATURES = ["pclass", "age", "family_size", "fare"]
CATEGORICAL_FEATURES = ["sex", "embarked"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])

## 12. Train a Random Forest classifier

A Random Forest combines many decision trees. The preprocessing and classifier are fitted together using only the training partition.

In [ ]:
evaluation_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        class_weight=None,
    )),
])
evaluation_model.fit(X_train, y_train)
print("Training complete.")

## 13. Evaluate on held-out passengers

Accuracy is the fraction classified correctly. The classification report shows precision, recall, and F1 score for each class. ROC AUC evaluates how well probabilities rank survivors above non-survivors. These metrics describe this split and may vary on other data.

In [ ]:
predictions = evaluation_model.predict(X_test)
probabilities = evaluation_model.predict_proba(X_test)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
print(f"ROC AUC:  {roc_auc_score(y_test, probabilities):.3f}")
print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=["Did not survive", "Survived"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions,
    display_labels=["Did not survive", "Survived"],
    cmap="Blues",
    values_format="d",
)
plt.title("Held-out test set confusion matrix")
plt.tight_layout()
plt.show()

## 14. Fit the final model and save it

After measuring performance, fit a fresh pipeline on all rows so the website demo can learn from the full practice dataset. This final fit is for the demo artifact; use the held-out results above when describing model performance. The saved feature names and pipeline steps match `app.py`.

In [ ]:
final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        class_weight=None,
    )),
])
final_model.fit(X, y)
joblib.dump(final_model, MODEL_PATH)
print(f"Saved model to: {MODEL_PATH.resolve()}")

## 15. Try a passenger prediction

`predict_proba` returns an estimated probability for each class. The value is a model estimate based on this historical practice dataset, not a guarantee or causal assessment.

In [ ]:
passenger = pd.DataFrame([{
    "pclass": 2,
    "sex": "female",
    "age": 28,
    "family_size": 2,
    "fare": 72.0,
    "embarked": "S",
}], columns=FEATURES)

survival_probability = final_model.predict_proba(passenger)[0, 1]
predicted_class = int(final_model.predict(passenger)[0])
print(f"Estimated survival chance: {survival_probability:.1%}")
print("Predicted class:", "Survived" if predicted_class == 1 else "Did not survive")

## Summary

This notebook explored the Titanic data, engineered family size, prepared numeric and categorical variables, trained a Random Forest pipeline, evaluated it on a held-out split, and saved the final pipeline as `titanic_survival_pipeline.joblib` for the website.
